# Chaotic N-Body CTM — Applied ML Assignment

This notebook is the Colab/local driver for the repository. The scientific goal is to study **how CTM internal dynamics evolve over thought ticks** on a hard three-way chaotic gravitational outcome task.

**Read `README.md`, `DATASET.md`, and `ANALYSIS_GUIDE.md` before interpreting results.**


In [ ]:
import os, sys, subprocess, pathlib
print(sys.version)
try:
    import torch
    print('torch', torch.__version__)
    print('CUDA', torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
        print('VRAM GiB', torch.cuda.get_device_properties(0).total_memory/1024**3)
except Exception as e:
    print(e)


## Project setup

If this notebook is opened **inside the extracted repository locally**, the next cell uses the current directory.

In Colab, upload `nbody_ctm.zip` to `/content/` first; the cell will extract it.


In [ ]:
from pathlib import Path
root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    zip_path = Path('/content/nbody_ctm.zip')
    if zip_path.exists():
        subprocess.run(['unzip','-q','-o',str(zip_path),'-d','/content'], check=True)
        root = Path('/content/nbody_ctm')
    elif (Path('/content/nbody_ctm') / 'pyproject.toml').exists():
        root = Path('/content/nbody_ctm')
    else:
        raise FileNotFoundError('Open notebook in repo locally, or upload /content/nbody_ctm.zip in Colab.')
os.chdir(root)
print('Project root:', root)
subprocess.run([sys.executable,'-m','pip','install','-e','.', '--no-build-isolation'], check=True)


## 1. Unit tests


In [ ]:
subprocess.run(['pytest'], check=True)


## 2. Generate the quick dataset

Do **not** jump directly to the main dataset. The quick configuration is the scientific sanity check.


In [ ]:
subprocess.run([sys.executable,'generate_data.py','--config','configs/quick.yaml'], check=True)


## 3. Audit for shortcuts


In [ ]:
subprocess.run([sys.executable,'inspect_data.py','--data','data/nbody_quick.npz'], check=True)


Inspect the printed shortcut results. If simple last-frame/handcrafted models are near-perfect, stop and revise the dataset before interpreting CTM dynamics.


## 4. Train quick CTM


In [ ]:
subprocess.run([sys.executable,'train.py','--config','configs/quick.yaml','--model','ctm','--seed','42','--out_dir','outputs/quick/ctm_seed42'], check=True)


## 5. Run internal-dynamics analysis


In [ ]:
subprocess.run([sys.executable,'analyze.py','--config','configs/quick.yaml','--checkpoint','outputs/quick/ctm_seed42/best_final.pt','--out_dir','outputs/quick/ctm_seed42/analysis'], check=True)


## 6. Inspect generated tables


In [ ]:
import pandas as pd, json
from pathlib import Path
a = Path('outputs/quick/ctm_seed42/analysis')
print(json.load(open(a/'summary.json')))
display(pd.read_csv(a/'per_tick_metrics.csv'))
display(pd.read_csv(a/'difficulty_summary.csv'))
display(pd.read_csv(a/'attention_event_bodies.csv').head(20))
display(pd.read_csv(a/'neuron_selectivity.csv').head(15))
if (a/'counterfactual_pair_dynamics.csv').exists():
    display(pd.read_csv(a/'counterfactual_pair_dynamics.csv').head(20))


## 7. Main experiment (run only after quick audit)

The cells below are intentionally not automatic. Freeze the data-generating process before running them.


In [ ]:
# MAIN DATASET — uncomment only after accepting the quick audit
# subprocess.run([sys.executable,'generate_data.py','--config','configs/main.yaml'], check=True)
# subprocess.run([sys.executable,'inspect_data.py','--data','data/nbody_main.npz'], check=True)


In [ ]:
# MAIN CTM SEEDS — uncomment when ready
# for seed in [42,123,2026]:
#     out = f'outputs/main/ctm_seed{seed}'
#     subprocess.run([sys.executable,'train.py','--config','configs/main.yaml','--model','ctm','--seed',str(seed),'--out_dir',out], check=True)
#     subprocess.run([sys.executable,'analyze.py','--config','configs/main.yaml','--checkpoint',f'{out}/best_final.pt','--out_dir',f'{out}/analysis'], check=True)


## Interpretation reminder

The assignment's main result is **not whether CTM beats a GRU**. Focus on how discrimination, certainty, calibration, attention, neuron diversity, synchronization, representation separation, settling time, and counterfactual-pair divergence change across CTM thought ticks.
